In [2]:
# ==============================================================================
# 1: 导入与全局配置
# ==============================================================================
import pandas as pd
import numpy as np
import os
# 替换为AdaBoost回归器
from sklearn.ensemble import AdaBoostRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr  # 新增：相关性计算

# ==============================================================================
# 2: 数据读取
# ==============================================================================
file_path = r"D:\project\数据整理.xlsx"
output_file = r"D:\project\AdaBoost预测结果_RMSE评分.xlsx"  # 文件名同步更新
sheet_as = "AS7343去除11条的异常数据"
sheet_ev = "EVERFINE去除11条的异常数据"

df_as = pd.read_excel(file_path, sheet_name=sheet_as)
df_ev = pd.read_excel(file_path, sheet_name=sheet_ev)

ids = df_as.iloc[:, 0].values
X = df_as.iloc[:, 1:].values
Y = df_ev.iloc[:, 1:].values

# ==============================================================================
# 3: 数据拆分与建模（更新为AdaBoost参数）
# ==============================================================================
# 数据拆分
indices = np.arange(len(ids))
stratify_labels = (ids > 343).astype(int)
X_tr, X_te, Y_tr, Y_te, idx_tr, idx_te = train_test_split(
    X, Y, indices, 
    test_size=0.2, 
    random_state=42,
    stratify=stratify_labels
)

# 定义AdaBoost固定参数（移除estimator__前缀，适配MultiOutputRegressor）
fixed_params = {
    'learning_rate': 0.1,
    'loss': 'linear',
    'n_estimators': 50,
    'random_state': 42  # 保留随机种子保证结果可复现
}

# 构建AdaBoost多输出回归模型
model_final = MultiOutputRegressor(AdaBoostRegressor(**fixed_params))
model_final.fit(X_tr, Y_tr)

# ==============================================================================
# 4: 模型评估
# ==============================================================================
# 测试集预测结果
Y_te_pred = model_final.predict(X_te)
# 计算测试集RMSE
score_rmse = np.sqrt(mean_squared_error(Y_te, Y_te_pred))
print(f"AdaBoost模型测试集 RMSE: {score_rmse:.4f}")

# ✅ 新增：光源光谱特征 ↔ AS7343 12通道 匹配性分析（核心）
# ==============================================================================
# 获取通道名
channel_names = list(df_as.columns[1:])
match_scores = []

# 计算每个通道与所有光谱波段的平均相关系数
for ch_idx in range(X.shape[1]):
    channel_data = X[:, ch_idx]
    corrs = []
    for band_idx in range(Y.shape[1]):
        spec_data = Y[:, band_idx]
#皮尔逊相关系数        
        r, _ = pearsonr(channel_data, spec_data)
        corrs.append(abs(r))
    avg_corr = np.mean(corrs)
    match_scores.append(avg_corr)

# 按匹配度从高到低排序输出
sorted_idx = np.argsort(match_scores)[::-1]
print("\n【通道匹配度排序（越高 → 与光谱特征越匹配）】")
for rank, idx in enumerate(sorted_idx, 1):
    ch_name = channel_names[idx]
    score = match_scores[idx]
    print(f"第{rank:2d}名 | {ch_name:4s} | 平均匹配系数 = {score:.4f}")

# 整体匹配度
print(f"\n【12通道整体平均匹配度】 = {np.mean(match_scores):.4f}")

AdaBoost模型测试集 RMSE: 0.0035

【通道匹配度排序（越高 → 与光谱特征越匹配）】
第 1名 | VIS  | 平均匹配系数 = 0.4908
第 2名 | FXL  | 平均匹配系数 = 0.4794
第 3名 | FY   | 平均匹配系数 = 0.4702
第 4名 | F6   | 平均匹配系数 = 0.4599
第 5名 | F7   | 平均匹配系数 = 0.4358
第 6名 | F5   | 平均匹配系数 = 0.4002
第 7名 | F2   | 平均匹配系数 = 0.3986
第 8名 | F8   | 平均匹配系数 = 0.3844
第 9名 | F1   | 平均匹配系数 = 0.3745
第10名 | F4   | 平均匹配系数 = 0.3712
第11名 | FZ   | 平均匹配系数 = 0.3697
第12名 | F3   | 平均匹配系数 = 0.3273

【12通道整体平均匹配度】 = 0.4135
